In [ ]:
import os
import sys
current = os.path.dirname(os.path.realpath("Visualize_Pairwise_Matches.ipynb"))
parent = os.path.dirname(current)
sys.path.append(parent)
# from transformers import CLIPProcessor, CLIPModel
import torch
import torch.hub
import torchvision
from torchvision.models import resnet50
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import clip
from PIL import Image
import requests
import torch.hub
import time
import pickle
import math

from match_utils import matching, stats, proggan, nethook, dataset, loading, plotting, layers, models, visualize_pairwisematch

In [ ]:
device = torch.device('cuda:3')

# 1. Load models

In [ ]:
gan_mode = "styleganxl"
discr_mode = "dino_vits16"

if gan_mode != "biggan" and gan_mode != "styleganxl":
    classidx = None
else: 
    classidx = 0

## Visualize over multiple models

In [ ]:
# Define the list of models to loop over
model_configs = [
    {
        "matches_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/matches/250724034859-57b156-30a20fd-code/matches/styleganxl_dinoshaders21k_cls0_1600",
        "discr_modelname": "dinoshaders21k",
        "discr_path": "/datadrive/shaden/project1/FeatureInversion/pretrained_models/dino_shaders21k.pth",
        "save_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/logs/styleganxl_dinoshaders21k_cls0_1600",
    },
    {
        "matches_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/matches/250724000907-7e6b1c-1939ea1-code/matches/styleganxl_dinoshadersmixup_cls0_1600",
        "discr_path": "/datadrive/shaden/project1/FeatureInversion/pretrained_models/dino_shaders_mixup.pth",
        "discr_modelname": "dinoshadersmixup",
        "save_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/logs/styleganxl_dinoshadersmixup_cls0_1600",
    },
    {
        "matches_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/matches/250724033931-cfd799-30a20fd-code/matches/styleganxl_dinoshaderskml_cls0_1600",
        "discr_modelname": "dinoshaderskml",
        "discr_path": "/datadrive/shaden/project1/FeatureInversion/pretrained_models/dino_shaders_kml.pth",
        "save_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/logs/styleganxl_dinoshaderskml_cls0_1600",
    },
    {
        "matches_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/matches/250724035349-8d780f-30a20fd-code/matches/styleganxl_dinoshaderskmlmixup_cls0_1600",
        "discr_modelname": "dinoshaderskmlmixup",
        "discr_path": "/datadrive/shaden/project1/FeatureInversion/pretrained_models/dino_vit_small_shaders_kml_mixup.pth",
        "save_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/logs/styleganxl_dinoshaderskmlmixup_cls0_1600",
    },
    {
        "matches_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/matches/250724174723-58736b-30a20fd-code/matches/styleganxl_dino_stylegan_cls0_1600",
        "discr_modelname": "dinostylegan",
        "discr_path": "/datadrive/shaden/project1/FeatureInversion/pretrained_models/dino_stylegan.pth",
        "save_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/logs/styleganxl_dino_stylegan_cls0_1600",
    },
    {
        "matches_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/matches/250724175244-1d9eb9-30a20fd-code/matches/styleganxl_dinodeadleaves_cls0_1600",
        "discr_modelname": "dinodeadleaves",
        "discr_path": "/datadrive/shaden/project1/FeatureInversion/pretrained_models/dino_dead_leaves.pth",
        "save_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/logs/styleganxl_dinodeadleaves_cls0_1600",
    },
    {
        "matches_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/matches/250724174320-58e229-30a20fd-code/matches/styleganxl_dinoimagenet_cls0_1600",
        "discr_modelname": "dinoimagenet",  
        "discr_path": "/datadrive/shaden/project1/FeatureInversion/pretrained_models/dino_imagenet.pth",
        "save_dir": "/datadrive/shaden/project1/kevin/shaders_rosetta/logs/styleganxl_dinoimagenet_cls0_1600",
    },    
]

In [ ]:
# start loop
# Loop over each model configuration
for config in model_configs:
    discr_path = config["discr_path"]
    discr_modelname = config["discr_modelname"]
    matches_dir = config["matches_dir"]
    
    print(f"\n{'='*80}")
    print(f"Processing model: {discr_modelname}")
    print(f"Model path: {discr_path}")
    print(f"Matches dir: {matches_dir}")
    print(f"{'='*80}\n")
    
    # Load models
    try:
        discr, discr_layers = models.load_discr(discr_mode, device, discr_path)
        gan, gan_layers = models.load_gan(gan_mode, device)
        
        ganlayers, discrlayers = layers.get_layers(
            gan,
            gan_layers=gan_layers, 
            discr=discr, 
            discr_layers=discr_layers,
            gan_mode=gan_mode, 
            discr_mode=discr_mode, 
            device=device
        )
        
        print("Loading...")
        table, gan_stats, discr_stats = loading.load_stats(root=matches_dir, device=device)
        print("Done")
        print(f"Table shape: {table.shape}")
        
        match_scores, _ = torch.max(table,1)
        
        n = 5
        _, top1_discr_per_gan = torch.topk(table,k=1,dim=1) # Gx1 
        _, topn_gan_per_discr = torch.topk(table,k=n, dim=0) # 5xD
        
        gan_match_units = []
        discr_match_units = []
        perfect_match_scores1 = []
        num_kmatches = 0 
        
        for gan_unit in range(table.shape[0]):
            top1_discr = top1_discr_per_gan[gan_unit].item()
            topn_gan = topn_gan_per_discr[:, top1_discr]
            
            for top_gan_for_discr in topn_gan:
                if top_gan_for_discr == gan_unit: # G unit j contained in its top D's top5
                    num_kmatches += 1
                    discr_match_units.append(top1_discr)
                    gan_match_units.append(gan_unit)
                    perfect_match_scores1.append(table[gan_unit, top1_discr])
                    break
        
        print(f"Number of matches: {len(gan_match_units)}")
        
        # Sort both lists by perfect_match_scores1 in descending order
        sorted_pairs = sorted(zip(perfect_match_scores1, gan_match_units), reverse=True)
        gan_match_units = [match for score, match in sorted_pairs]
        scores = [score.item() for score, _ in sorted_pairs]
        
        # Load dataset
        batch_size = 1
        epochs = 1
        z_dataset, c_dataset = dataset.create_dataset(gan, gan_mode, batch_size, epochs, classidx, device)
        
        # Visualize matches
        save_dir = config["save_dir"]
        num_units_to_viz = 5
        visualize_pairwisematch.viz_matches(table, gan, discr, (z_dataset,c_dataset), ganlayers, discrlayers, gan_stats, discr_stats, gan_mode, discr_mode, gan_match_units[:num_units_to_viz], scores[:num_units_to_viz], gan_name=gan_mode, discr_name=discr_modelname, save_dir=save_dir)

    except Exception as e:
        print(f"Error processing {discr_modelname}: {str(e)}")
        continue

## Analyze 1 by 1

In [ ]:
discr_path = "/datadrive/shaden/project1/FeatureInversion/pretrained_models/dino_shaders_mixup.pth"
discr_modelname = "dinoshadersmixup"

In [ ]:
discr, discr_layers = models.load_discr(discr_mode, device, discr_path)
gan, gan_layers = models.load_gan(gan_mode, device)

ganlayers, discrlayers = layers.get_layers(gan,
                                        gan_layers=gan_layers, 
                                        discr=discr, 
                                        discr_layers=discr_layers,
                                        gan_mode=gan_mode, 
                                        discr_mode=discr_mode, 
                                        device=device)



In [ ]:
discrlayers

# 2. Load tables

In [ ]:
# SPECIFY MATCH DIRECTORY
matches_dir = "/datadrive/shaden/project1/kevin/shaders_rosetta/matches/250724000907-7e6b1c-1939ea1-code/matches/styleganxl_dinoshadersmixup_cls0_1600"

In [ ]:
table, gan_stats, discr_stats = loading.load_stats(root=matches_dir, device=device)

In [ ]:
table.shape

# 3. Compute matches

In [ ]:
match_scores, _ = torch.max(table,1)

In [ ]:
n = 5
_, top1_discr_per_gan = torch.topk(table,k=1,dim=1) # Gx1 
_, topn_gan_per_discr = torch.topk(table,k=n, dim=0) # 5xD

In [ ]:
gan_match_units = []
discr_match_units = []

perfect_match_scores1= []

num_kmatches = 0 

for gan_unit in range(table.shape[0]):
    top1_discr = top1_discr_per_gan[gan_unit].item()
    topn_gan = topn_gan_per_discr[:, top1_discr]
    
    for top_gan_for_discr in topn_gan:
        if top_gan_for_discr == gan_unit: # G unit j contained in its top D's top5
            num_kmatches += 1
            discr_match_units.append(top1_discr)

            gan_match_units.append(gan_unit)
            perfect_match_scores1.append(table[gan_unit, top1_discr])
            break

print(len(gan_match_units))

#### Sort units according to scores

In [ ]:
# Sort both lists by perfect_match_scores1 in descending order
sorted_pairs = sorted(zip(perfect_match_scores1, gan_match_units), reverse=True)

gan_match_units = [match for score, match in sorted_pairs]
scores = [score.item() for score, _ in sorted_pairs]

# 4. Visualize Matches over Sample Images

In [ ]:
batch_size = 1
epochs = 1
classidx = None
z_dataset, c_dataset = dataset.create_dataset(gan, gan_mode, batch_size, epochs, classidx, device)

In [ ]:
init = gan(z_dataset,c_dataset,1)
init = (init.detach().cpu()+1)/2
init = torch.permute(init[0], (1,2,0))
plt.imshow(init)
init = (init-torch.min(init))/(torch.max(init)-torch.min(init))
init = np.uint8(init*255)

In [ ]:
num_units_to_viz = 2

In [ ]:
visualize_pairwisematch.viz_matches(table, gan, discr, (z_dataset,c_dataset), ganlayers, discrlayers, gan_stats, discr_stats, gan_mode, discr_mode, gan_match_units[:num_units_to_viz], scores[:num_units_to_viz])